# TetheredAI MLB Model Lab — Statcast Feature Upgrade

This notebook is designed to help you move beyond team-form baselines by evaluating leakage-safe Statcast-derived features: contact quality, starter pitch quality, pitch mix, bullpen contact/whiff quality, and offense-vs-pitcher-hand.

Primary selection metrics: **log loss**, **Brier score**, calibration, and AUC. Accuracy is shown but should not be the primary promotion criterion.

Lineup features are intentionally excluded.

In [ ]:
from pathlib import Path
import sys, json, warnings, platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Locate project root whether running from notebooks/ or project root.
cwd = Path.cwd().resolve()
if cwd.name == 'notebooks':
    PROJECT_ROOT = cwd.parent
elif (cwd / 'src').exists():
    PROJECT_ROOT = cwd
else:
    PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'src').exists()), cwd)

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('PROJECT_ROOT:', PROJECT_ROOT)
print('Python:', platform.python_version())

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import ParameterGrid, ParameterSampler
from sklearn.metrics import (
    log_loss, brier_score_loss, roc_auc_score, accuracy_score,
    classification_report, confusion_matrix
)
from sklearn.inspection import permutation_importance
from sklearn.calibration import CalibratedClassifierCV
import joblib

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception as exc:
    HAS_XGB = False
    print('XGBoost unavailable:', exc)

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception as exc:
    HAS_LGBM = False
    print('LightGBM unavailable:', exc)

from mlb_betting.feature_engineering import get_model_feature_columns
try:
    from mlb_betting.statcast_features import get_statcast_feature_columns
except Exception:
    get_statcast_feature_columns = lambda frame: []

## Load feature data

Run the GitHub full feature refresh first, then pull the repo. The feature file should already include the Statcast-derived columns if you ran `09_fetch_statcast.py` before `03_build_features.py`.

In [ ]:
FEATURE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'mlb_game_features.parquet'
PRED_PATH = PROJECT_ROOT / 'data' / 'predictions' / 'mlb_moneyline_predictions.csv'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(exist_ok=True, parents=True)

features = pd.read_parquet(FEATURE_PATH)
features['game_datetime_utc'] = pd.to_datetime(features['game_datetime_utc'], utc=True, errors='coerce')
features['official_date'] = pd.to_datetime(features['official_date'], errors='coerce')
print(features.shape)
display(features.tail())

In [ ]:
completed = features[features['target_home_win'].notna()].copy()
completed = completed.sort_values(['game_datetime_utc', 'game_pk']).reset_index(drop=True)
completed['target_home_win'] = completed['target_home_win'].astype(int)
print('completed:', completed.shape)
print('date range:', completed['game_datetime_utc'].min(), 'to', completed['game_datetime_utc'].max())
print('home win rate:', completed['target_home_win'].mean())

## Missingness and feature groups

In [ ]:
all_model_cols = get_model_feature_columns(completed, include_market=False, min_non_null_rate=0.02)
statcast_cols = get_statcast_feature_columns(completed)
print('model cols:', len(all_model_cols))
print('statcast cols:', len(statcast_cols))

missing = completed[all_model_cols].isna().mean().sort_values(ascending=False)
display(missing.head(60).to_frame('missing_pct'))

In [ ]:
def cols_containing(tokens, base_cols=None):
    base_cols = list(base_cols or all_model_cols)
    if isinstance(tokens, str):
        tokens = [tokens]
    return [c for c in base_cols if any(t.lower() in c.lower() for t in tokens)]

elo_cols = cols_containing('elo')
starter_box_cols = [c for c in all_model_cols if 'starter_' in c.lower() and 'statcast' not in c.lower()]
starter_sc_cols = cols_containing('starter_statcast')
bullpen_box_cols = [c for c in all_model_cols if 'bullpen' in c.lower() and 'statcast' not in c.lower() and '_sc_' not in c.lower()]
bullpen_sc_cols = cols_containing(['bullpen_sc', 'sc_bullpen'])
team_form_cols = cols_containing(['win_last', 'run_diff', 'runs_for', 'runs_against', 'season_to_date', 'rest_days', 'games_played'])
team_box_cols = cols_containing(['team_box', 'team_off_'])
team_vs_hand_cols = cols_containing(['team_vs_hand'])
statcast_team_cols = cols_containing(['team_off_sc', 'team_vs_hand_sc'])

FEATURE_SETS = {
    'baseline_team_form': sorted(set(team_form_cols + elo_cols)),
    'elo_only': sorted(set(elo_cols)),
    'starter_boxscore_only': sorted(set(starter_box_cols + elo_cols)),
    'starter_statcast_only': sorted(set(starter_sc_cols + elo_cols)),
    'starter_all': sorted(set(starter_box_cols + starter_sc_cols + elo_cols)),
    'bullpen_all': sorted(set(bullpen_box_cols + bullpen_sc_cols + elo_cols)),
    'statcast_team_only': sorted(set(statcast_team_cols + elo_cols)),
    'enhanced_no_market': sorted(set(all_model_cols)),
}

for name, cols in FEATURE_SETS.items():
    print(f'{name:25s}', len(cols))

## Time split and baselines

In [ ]:
HOLDOUT_GAMES = 800
MIN_NON_NULL_RATE = 0.02

train = completed.iloc[:-HOLDOUT_GAMES].copy()
holdout = completed.iloc[-HOLDOUT_GAMES:].copy()

y_train = train['target_home_win'].astype(int)
y_holdout = holdout['target_home_win'].astype(int)
print('train:', train.shape, train['game_datetime_utc'].min(), train['game_datetime_utc'].max())
print('holdout:', holdout.shape, holdout['game_datetime_utc'].min(), holdout['game_datetime_utc'].max())
print('train home rate:', y_train.mean())
print('holdout home rate:', y_holdout.mean())

In [ ]:
def score_probs(y_true, prob, label):
    prob = np.clip(np.asarray(prob, dtype=float), 1e-6, 1-1e-6)
    pred = (prob >= 0.5).astype(int)
    return {
        'model_key': label,
        'n': len(y_true),
        'log_loss': log_loss(y_true, prob),
        'brier': brier_score_loss(y_true, prob),
        'accuracy_50pct': accuracy_score(y_true, pred),
        'avg_pred': float(np.mean(prob)),
        'actual_rate': float(np.mean(y_true)),
        'roc_auc': roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else np.nan,
    }

baselines = []
train_home_rate = y_train.mean()
baselines.append(score_probs(y_holdout, np.repeat(train_home_rate, len(y_holdout)), 'baseline__train_home_rate'))
baselines.append(score_probs(y_holdout, np.ones(len(y_holdout)) * 0.500001, 'baseline__coinflip'))
if 'elo_home_win_prob' in holdout.columns:
    baselines.append(score_probs(y_holdout, holdout['elo_home_win_prob'].fillna(train_home_rate), 'baseline__elo_home_win_prob'))

baseline_df = pd.DataFrame(baselines).sort_values('log_loss')
display(baseline_df)

## Model candidates

Use the quick search first. Turn on full search only for finalists.

In [ ]:
RUN_FULL_SEARCH = False
RUN_SVM = False
MODEL_NAMES = ['logit_l2', 'random_forest', 'extra_trees', 'hist_gradient_boosting', 'xgboost', 'lightgbm']
if RUN_SVM:
    MODEL_NAMES.append('svm_rbf')

RANDOM_STATE = 42

def make_pipeline(model_name, params=None):
    params = params or {}
    if model_name == 'logit_l2':
        model = LogisticRegression(max_iter=5000, solver='lbfgs', **params)
        return Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)), ('scaler', StandardScaler()), ('model', model)])
    if model_name == 'random_forest':
        model = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **params)
        return Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)), ('model', model)])
    if model_name == 'extra_trees':
        model = ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1, **params)
        return Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)), ('model', model)])
    if model_name == 'hist_gradient_boosting':
        model = HistGradientBoostingClassifier(random_state=RANDOM_STATE, **params)
        return Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)), ('model', model)])
    if model_name == 'svm_rbf':
        model = SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE, **params)
        return Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)), ('scaler', StandardScaler()), ('model', model)])
    if model_name == 'xgboost':
        if not HAS_XGB:
            return None
        model = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', n_jobs=-1, **params)
        return Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)), ('model', model)])
    if model_name == 'lightgbm':
        if not HAS_LGBM:
            return None
        model = LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1, **params)
        return Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)), ('model', model)])
    raise ValueError(model_name)

QUICK_GRIDS = {
    'logit_l2': [{'model__C': c, 'model__class_weight': cw} for c in [0.1, 0.5, 1.0, 2.0] for cw in [None, 'balanced']],
    'random_forest': [{'model__n_estimators': n, 'model__max_depth': d, 'model__min_samples_leaf': leaf, 'model__max_features': mf} for n in [300, 500] for d in [3, 4, 6] for leaf in [25, 50] for mf in ['sqrt']],
    'extra_trees': [{'model__n_estimators': n, 'model__max_depth': d, 'model__min_samples_leaf': leaf, 'model__max_features': mf} for n in [300, 500] for d in [3, 4, 6] for leaf in [25, 50] for mf in ['sqrt']],
    'hist_gradient_boosting': [{'model__learning_rate': lr, 'model__max_iter': it, 'model__max_leaf_nodes': leaves, 'model__l2_regularization': l2} for lr in [0.01, 0.03, 0.05] for it in [100, 200] for leaves in [7, 15] for l2 in [0.0, 0.1, 1.0]],
    'svm_rbf': [{'model__C': c, 'model__gamma': g, 'model__class_weight': cw} for c in [0.25, 0.5, 1.0] for g in ['scale', 0.01, 0.03] for cw in [None, 'balanced']],
    'xgboost': [{'model__n_estimators': n, 'model__max_depth': d, 'model__learning_rate': lr, 'model__min_child_weight': mcw, 'model__subsample': ss, 'model__colsample_bytree': cs, 'model__reg_lambda': l2, 'model__reg_alpha': l1} for n in [200, 300] for d in [2, 3] for lr in [0.01, 0.02, 0.05] for mcw in [5, 10] for ss in [0.8, 0.9] for cs in [0.8, 0.9] for l2 in [1.0, 2.0] for l1 in [0.0, 0.1]],
    'lightgbm': [{'model__n_estimators': n, 'model__num_leaves': leaves, 'model__max_depth': d, 'model__learning_rate': lr, 'model__min_child_samples': mcs, 'model__subsample': ss, 'model__colsample_bytree': cs, 'model__reg_lambda': l2} for n in [200, 300] for leaves in [7, 15, 31] for d in [2, 3, -1] for lr in [0.01, 0.03, 0.05] for mcs in [25, 50] for ss in [0.8, 0.9] for cs in [0.8, 0.9] for l2 in [0.0, 1.0, 2.0]],
}

MAX_PARAM_TRIALS = 36 if not RUN_FULL_SEARCH else 120
print('models:', MODEL_NAMES)

In [ ]:
def set_pipeline_params(pipe, param_dict):
    pipe.set_params(**param_dict)
    return pipe

def eval_model(feature_set_name, cols, model_name, params):
    cols = [c for c in cols if c in completed.columns and pd.api.types.is_numeric_dtype(completed[c])]
    cols = [c for c in cols if train[c].notna().mean() >= MIN_NON_NULL_RATE]
    if len(cols) == 0:
        return None
    pipe = make_pipeline(model_name)
    if pipe is None:
        return None
    pipe = set_pipeline_params(pipe, params)
    X_train = train[cols]
    X_hold = holdout[cols]
    pipe.fit(X_train, y_train)
    prob = pipe.predict_proba(X_hold)[:, 1]
    metrics = score_probs(y_holdout, prob, f'{feature_set_name}__{model_name}')
    metrics.update({
        'feature_set': feature_set_name,
        'model_name': model_name,
        'feature_count': len(cols),
        'params': params,
        'estimator': pipe,
        'feature_cols': cols,
        'probs': prob,
    })
    return metrics

results = []
for fs_name, cols in FEATURE_SETS.items():
    if len(cols) == 0:
        continue
    for model_name in MODEL_NAMES:
        if model_name == 'xgboost' and not HAS_XGB:
            continue
        if model_name == 'lightgbm' and not HAS_LGBM:
            continue
        grid = QUICK_GRIDS.get(model_name, [{}])
        if len(grid) > MAX_PARAM_TRIALS:
            rng = np.random.default_rng(RANDOM_STATE)
            idx = rng.choice(len(grid), size=MAX_PARAM_TRIALS, replace=False)
            param_list = [grid[i] for i in idx]
        else:
            param_list = grid
        print(f'Running {fs_name} / {model_name} / {len(param_list)} trials / {len(cols)} cols')
        best = None
        for params in param_list:
            out = eval_model(fs_name, cols, model_name, params)
            if out is None:
                continue
            if best is None or out['log_loss'] < best['log_loss']:
                best = out
        if best is not None:
            print('  best logloss:', best['log_loss'], 'brier:', best['brier'], 'auc:', best['roc_auc'])
            results.append(best)

res_df = pd.DataFrame([{k:v for k,v in r.items() if k not in {'estimator','feature_cols','probs'}} for r in results])
res_df = pd.concat([baseline_df, res_df], ignore_index=True, sort=False)
display(res_df.sort_values(['log_loss','brier']).head(30))

## Calibration diagnostics

In [ ]:
best_row = min(results, key=lambda r: r['log_loss'])
print('Best model:', best_row['model_key'])
print('Feature count:', best_row['feature_count'])
print('Metrics:', {k: best_row[k] for k in ['log_loss','brier','accuracy_50pct','roc_auc','avg_pred','actual_rate']})
print('Best params:', best_row['params'])

prob = best_row['probs']
pred = (prob >= 0.5).astype(int)
print(classification_report(y_holdout, pred, digits=3))
print(confusion_matrix(y_holdout, pred))

In [ ]:
cal = pd.DataFrame({'y': y_holdout.values, 'prob': prob})
cal['prob_bucket'] = pd.cut(cal['prob'], bins=[0, .35, .4, .45, .5, .55, .6, .65, .7, 1.0], include_lowest=True)
cal_table = cal.groupby('prob_bucket').agg(
    games=('y','size'),
    avg_pred_prob=('prob','mean'),
    actual_home_win_rate=('y','mean'),
)
cal_table['calibration_error'] = cal_table['actual_home_win_rate'] - cal_table['avg_pred_prob']
display(cal_table)

plt.figure(figsize=(6,5))
plt.plot([0,1],[0,1], linestyle='--')
plt.scatter(cal_table['avg_pred_prob'], cal_table['actual_home_win_rate'], s=np.maximum(cal_table['games'].fillna(0), 1) * 4)
plt.xlabel('Average predicted probability')
plt.ylabel('Actual home win rate')
plt.title(f'Calibration: {best_row["model_key"]}')
plt.grid(True, alpha=0.3)
plt.show()

## Permutation importance for the best model

In [ ]:
X_hold = holdout[best_row['feature_cols']]
perm = permutation_importance(
    best_row['estimator'], X_hold, y_holdout,
    scoring='neg_log_loss', n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1
)
imp = pd.DataFrame({
    'feature': best_row['feature_cols'],
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std,
}).sort_values('importance_mean', ascending=False)
display(imp.head(40))

## Optional calibration layer

Use this when a model has good ranking but poor probability calibration. This simple split uses the tail of the training period as calibration data. For final governance, use walk-forward or nested calibration.

In [ ]:
CAL_SIZE = min(400, max(100, int(len(train) * 0.15)))
train_fit = train.iloc[:-CAL_SIZE].copy()
train_cal = train.iloc[-CAL_SIZE:].copy()
y_fit = train_fit['target_home_win'].astype(int)
y_cal = train_cal['target_home_win'].astype(int)
cols = best_row['feature_cols']

base_est = make_pipeline(best_row['model_name'])
base_est.set_params(**best_row['params'])
base_est.fit(train_fit[cols], y_fit)

for method in ['sigmoid', 'isotonic']:
    try:
        cal_model = CalibratedClassifierCV(base_est, method=method, cv='prefit')
        cal_model.fit(train_cal[cols], y_cal)
        cal_prob = cal_model.predict_proba(holdout[cols])[:, 1]
        print(method, score_probs(y_holdout, cal_prob, f'calibrated__{method}'))
    except Exception as exc:
        print(method, 'failed:', exc)

## Edge/EV tuning when historical odds exist

This section only works when completed holdout games have market odds fields populated.

In [ ]:
def american_profit_per_unit(price):
    price = float(price)
    if price > 0:
        return price / 100.0
    return 100.0 / abs(price)

market_cols = ['market_home_no_vig_prob','market_away_no_vig_prob','home_moneyline_median','away_moneyline_median']
if all(c in holdout.columns for c in market_cols) and holdout[market_cols].notna().all(axis=1).sum() > 50:
    bt = holdout.copy()
    bt['model_home_prob'] = prob
    bt['model_away_prob'] = 1 - prob
    bt['edge_home'] = bt['model_home_prob'] - bt['market_home_no_vig_prob']
    bt['edge_away'] = bt['model_away_prob'] - bt['market_away_no_vig_prob']
    rows = []
    for min_edge in [0.00, 0.01, 0.02, 0.03, 0.04, 0.05]:
        for side in ['home','away']:
            edge_col = f'edge_{side}'
            price_col = f'{side}_moneyline_median'
            model_col = f'model_{side}_prob'
            bets = bt[(bt[edge_col] >= min_edge) & bt[price_col].notna()].copy()
            if bets.empty:
                continue
            if side == 'home':
                won = bets['target_home_win'].astype(int).values
            else:
                won = 1 - bets['target_home_win'].astype(int).values
            profits = []
            for w, price in zip(won, bets[price_col]):
                profits.append(american_profit_per_unit(price) if w else -1.0)
            rows.append({
                'min_edge': min_edge, 'side': side, 'bets': len(bets),
                'win_rate': np.mean(won), 'avg_model_prob': bets[model_col].mean(),
                'roi_per_unit': np.mean(profits), 'total_units': np.sum(profits),
            })
    display(pd.DataFrame(rows).sort_values(['min_edge','side']))
else:
    print('Not enough completed games with historical market odds yet. Keep collecting odds snapshots.')

## Manual champion export

Only set `APPROVE_EXPORT=True` after you have reviewed log loss, Brier, calibration, and feature importance.

In [ ]:
APPROVE_EXPORT = False
CHAMPION_NOTES = 'Statcast feature model selected manually after notebook review.'

def export_champion_model(row, notes=CHAMPION_NOTES):
    created_at = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    champion_path = MODEL_DIR / 'mlb_moneyline_champion.joblib'
    metadata_path = MODEL_DIR / 'mlb_moneyline_champion_metadata.json'
    archive_dir = MODEL_DIR / 'archive'
    archive_dir.mkdir(parents=True, exist_ok=True)
    archive_path = archive_dir / f'mlb_moneyline_{row["model_key"]}_{timestamp}.joblib'.replace('/','_')
    metadata = {
        'model_name': 'mlb_moneyline_champion',
        'model_key': row['model_key'],
        'feature_set': row['feature_set'],
        'model_family': row['model_name'],
        'feature_count': len(row['feature_cols']),
        'feature_cols': list(row['feature_cols']),
        'params': row['params'],
        'metrics': {k: float(row[k]) for k in ['log_loss','brier','accuracy_50pct','roc_auc','avg_pred','actual_rate']},
        'notes': notes,
        'created_at_utc': created_at,
    }
    bundle = {
        'model_name': 'mlb_moneyline_champion',
        'estimator': row['estimator'],
        'feature_cols': list(row['feature_cols']),
        'target_col': 'target_home_win',
        'model_family': row['model_name'],
        'feature_set_name': row['feature_set'],
        'metrics': metadata['metrics'],
        'metadata': metadata,
        'created_at_utc': created_at,
    }
    joblib.dump(bundle, champion_path)
    joblib.dump(bundle, archive_path)
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    print('Saved:', champion_path)
    print('Saved:', metadata_path)
    print('Archived:', archive_path)

if APPROVE_EXPORT:
    export_champion_model(best_row)
else:
    print('APPROVE_EXPORT is False. Review results before exporting champion model.')